# 🔧 Module 01: dbt (Data Build Tool) Fundamentals & Architecture

Welcome to **Module 01** of **Project 11: dbt for Production Analytics Engineering**.

In this notebook, we will explore:
1. **Core dbt Architecture**: How dbt compiles Jinja SQL templates into native warehouse SQL statements.
2. **The `ref()` and `source()` Macros**: Building dynamic Lineage Directed Acyclic Graphs (DAGs).
3. **Jinja2 Templating**: Writing dynamic SQL logic using control loops and macros.
4. **Programmatic dbt Execution**: Running dbt operations via Python wrappers.
5. **Artifact Analysis**: Parsing `manifest.json` and `run_results.json` for metadata inspection.

In [ ]:
import os
import json
import yaml
from pathlib import Path

# Define project path
PROJECT_ROOT = Path("..").resolve()
DBT_PROJECT_DIR = PROJECT_ROOT / "src" / "dbt_project"
print(f"dbt Project Directory: {DBT_PROJECT_DIR}")
assert (DBT_PROJECT_DIR / "dbt_project.yml").exists(), "dbt_project.yml must exist!"

## 1. Inspecting `dbt_project.yml`

`dbt_project.yml` is the primary entry point for any dbt project. It specifies:
- Project name, version, profile
- Model directory defaults and materialization strategies per layer (`staging`, `intermediate`, `marts`)
- Seed column configurations and global variables

In [ ]:
# Load and inspect dbt_project.yml
with open(DBT_PROJECT_DIR / "dbt_project.yml", "r") as f:
    dbt_config = yaml.safe_load(f)

print("Project Name:", dbt_config.get("name"))
print("Target Profile:", dbt_config.get("profile"))
print("Global Variables:", json.dumps(dbt_config.get("vars", {}), indent=2))
print("Model Layer Materializations:")
print(yaml.dump(dbt_config.get("models", {})))

## 2. Understanding Jinja Compilation & `ref()` Resolution

Let's simulate how Jinja compiles `{{ ref('stg_orders') }}` and `{{ source('ecom_raw', 'raw_customers') }}` into warehouse physical SQL queries.

In [ ]:
from jinja2 import Template

# Sample dbt SQL model template
sample_sql_template = """
WITH customers AS (
    SELECT * FROM {{ source('ecom_raw', 'raw_customers') }}
),
orders AS (
    SELECT * FROM {{ ref('stg_orders') }}
)
SELECT
    c.id AS customer_id,
    COUNT(o.order_id) AS total_orders
FROM customers c
LEFT JOIN orders o ON c.id = o.customer_id
GROUP BY 1
"""

# Mock dbt macro resolution functions
def mock_source(source_name, table_name):
    return f"analytics_db.{source_name}.{table_name}"

def mock_ref(model_name):
    return f"analytics_db.dbt_staging.{model_name}"

template = Template(sample_sql_template)
compiled_sql = template.render(source=mock_source, ref=mock_ref)
print("--- COMPILED SQL OUTPUT ---")
print(compiled_sql)

## 3. Custom Jinja Macro Simulation: Hashed Surrogate Key Generation

In distributed data warehouses, surrogate keys are created by hashing primary business attributes using MD5.

In [ ]:
import hashlib

def generate_surrogate_key(values: list[str]) -> str:
    """Simulates dbt_utils.generate_surrogate_key logic."""
    cleansed = [str(v) if v is not None else "_dbt_null_" for v in values]
    concat_str = "-".join(cleansed)
    return hashlib.md5(concat_str.encode("utf-8")).hexdigest()

# Generate sample surrogate key for order item
surrogate_key = generate_surrogate_key(["ORD_10024", "PROD_8841"])
print(f"Order Item Surrogate Key: {surrogate_key}")
assert len(surrogate_key) == 32, "MD5 surrogate key must be 32 characters!"

## 4. Programmatic dbt Execution & Output Validation

We can invoke our custom Python runner `src/scripts/run_dbt.py` to trigger dbt runs and validate execution output programmatically.

In [ ]:
from scripts.run_dbt import run_dbt_command

# Execute dbt debug to verify credentials and project paths
return_code, stdout, duration = run_dbt_command(["debug"], str(DBT_PROJECT_DIR))
print(f"dbt debug finished in {duration:.2f}s with exit code {return_code}")
print(stdout[:400])  # Output preview

## 5. Summary & Next Steps

In this notebook, we learned how dbt resolves models into a compiled DAG, creates hashed surrogate keys, and provides programmatic Python control hooks.

Proceed to inspect the model files in `src/dbt_project/models/` and execute unit tests via `pytest`.